In [ ]:

# ===================== 1. 导入工具库 =====================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
import zipfile
import io
from openpyxl import Workbook
from openpyxl.drawing.image import Image
warnings.filterwarnings('ignore')  # 忽略警告
# ===================== 2. 基础配置 =====================
# ROOT_FOLDER可以是ZIP文件路径
ROOT_FOLDER = r"/mnt/c/Users/M259941/Desktop/Desktop/----400.Inbox/temp"
# 初始化结果列表
results = []
charts_data = []  # 存储图表数据和相关信息
# ===================== 3. 定义函数来获取所有CSV文件 =====================
def get_all_csv_files(root):
    """从文件夹或ZIP文件中获取所有CSV文件"""
    csv_files = []
    
    # 检查是否是ZIP文件
    if root.lower().endswith('.zip'):
        print(f"检测到ZIP文件: {root}")
        
        try:
            with zipfile.ZipFile(root, 'r') as zip_ref:
                # 列出ZIP文件中的所有文件
                zip_file_list = zip_ref.namelist()
                print(f"ZIP文件中包含 {len(zip_file_list)} 个文件")
                
                # 筛选出CSV文件
                for file_name in zip_file_list:
                    if file_name.lower().endswith('.csv'):
                        # 存储ZIP文件信息和文件名的元组
                        csv_files.append((root, file_name))
                        print(f"  找到CSV文件: {file_name}")
        
        except zipfile.BadZipFile:
            print(f"错误: {root} 不是有效的ZIP文件")
        except FileNotFoundError:
            print(f"错误: 找不到文件 {root}")
        except Exception as e:
            print(f"打开ZIP文件时出错: {str(e)}")
    else:
        # 如果是文件夹，使用原来的方法
        print(f"检测到文件夹: {root}")
        for dirpath, _, filenames in os.walk(root):
            for f in filenames:
                if f.lower().endswith('.csv'):  # 添加 .lower() 处理大小写
                    csv_files.append(os.path.join(dirpath, f))
    
    return csv_files
# 获取所有要处理的CSV文件
file_list = get_all_csv_files(ROOT_FOLDER)
print(f"找到 {len(file_list)} 个CSV文件")
# 在文件遍历后添加
if not file_list:
    print(f"错误：在 {ROOT_FOLDER} 中未找到CSV文件")
    
    # 如果是ZIP文件，尝试列出内容
    if ROOT_FOLDER.lower().endswith('.zip'):
        try:
            with zipfile.ZipFile(ROOT_FOLDER, 'r') as zip_ref:
                print("\nZIP文件中的内容：")
                for file_name in zip_ref.namelist():
                    print(f"  - {file_name}")
        except:
            print(f"无法打开ZIP文件: {ROOT_FOLDER}")
# ===================== 4. 批量处理每个文件 =====================
for file_item in file_list:
    try:
        # 判断是否是ZIP文件中的文件
        if isinstance(file_item, tuple) and len(file_item) == 2:
            # 来自ZIP文件
            zip_path, file_in_zip = file_item
            file_name = os.path.basename(file_in_zip)  # 只获取文件名，不包含路径
            print(f"正在处理ZIP中的文件：{file_name}")
            
            # 从ZIP文件中读取CSV
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                with zip_ref.open(file_in_zip) as csv_file:
                    # 读取CSV文件内容
                    csv_content = csv_file.read()
                    # 将字节内容转换为字符串
                    csv_text = csv_content.decode('cp1252', errors='ignore')
                    # 使用pandas读取CSV
                    df = pd.read_csv(io.StringIO(csv_text), skiprows=12)
        else:
            # 来自普通文件夹
            file_path = file_item
            file_name = os.path.basename(file_path)
            print(f"正在处理：{file_name}")
            
            # 读取CSV文件
            df = pd.read_csv(file_path, skiprows=12, encoding="cp1252")
        
        total_rows = len(df)
        
        if total_rows < 13:
            print(f"  ⚠️ 文件 {file_name} 行数不足13行，跳过")
            continue
        
        # 提取数据
        x_data = df.iloc[12:, 0].values
        y_data = df.iloc[12:, 6].values
        
        # 检查列数是否足够
        if df.shape[1] < 7:
            print(f"  ⚠️ 文件 {file_name} 列数不足7列，跳过")
            continue
        
        # 转换为数值类型（处理可能的非数值数据）
        x_data = pd.to_numeric(x_data, errors='coerce')
        y_data = pd.to_numeric(y_data, errors='coerce')
        
        # 移除NaN值
        mask = ~(np.isnan(x_data) | np.isnan(y_data))
        x_data = x_data[mask]
        y_data = y_data[mask]
        
        if len(x_data) < 2 or len(y_data) < 2:
            print(f"  ⚠️ 文件 {file_name} 有效数据不足，跳过")
            continue
        
        # ===================== 5. 线性回归计算 =====================
        reg_start_idx = int(len(x_data) * 2/3)
        if reg_start_idx >= len(x_data) - 1:
            reg_start_idx = len(x_data) // 2
        
        x_reg = x_data[reg_start_idx:]
        y_reg = y_data[reg_start_idx:]
        
        # 线性回归
        if len(x_reg) < 2:
            print(f"  ⚠️ 回归数据不足，跳过")
            continue
            
        slope, intercept, _, _, _ = stats.linregress(x_reg, y_reg)
        
        # 防止除零
        if abs(slope) < 1e-10:
            t197 = np.nan
            t195 = np.nan
        else:
            t197 = (97 - intercept) / slope
            t195 = (95 - intercept) / slope
        
        trtime = x_data[-1] if len(x_data) > 0 else np.nan
        trlt = y_data[-1] if len(y_data) > 0 else np.nan
        
        # 提取E列数据，注意异常处理
        try:
            ltcd = df.iloc[12, 4] if df.shape[1] > 4 else np.nan
        except:
            ltcd = np.nan
        
        # ===================== 6. 创建散点图并保存到内存 =====================
        # 创建散点图
        fig, ax = plt.subplots(figsize=(3, 2))  # 设置较小尺寸，适合单元格内显示
        ax.scatter(x_data, y_data, c='red', s=3, label='数据点')
        ax.plot(x_data, y_data, 'b-', linewidth=1, alpha=0.7)  # 添加连线
        
        # 绘制回归线
        #if not np.isnan(slope) and not np.isnan(intercept):
        #    ax.plot(x_reg, slope * x_reg + intercept, 'g--', 
        #           linewidth=1, label='回归线', alpha=0.7)
        
        # 设置标题
        #try:
            #title = str(df.iloc[2, 0]) if pd.notna(df.iloc[2, 0]) else file_name
        title = file_name
        #except:
            #title = file_name
        
        # 简化标题，只显示文件名
        short_title = file_name[:20] + "..." if len(file_name) > 20 else file_name
        
        ax.set_title(short_title, fontsize=8)
        
        # 简化坐标轴标签
        ax.set_xlabel('X', fontsize=6)
        ax.set_ylabel('Y', fontsize=6)
        
        # 设置坐标轴字体大小
        ax.tick_params(axis='both', which='major', labelsize=6)
        
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        # 将图表保存到内存
        img_data = io.BytesIO()
        fig.savefig(img_data, format='png', dpi=100, bbox_inches='tight')
        img_data.seek(0)
        
        # 关闭图表释放内存
        plt.close(fig)
        
        # 存储图表数据
        charts_data.append({
            'file_name': file_name,
            'img_data': img_data,
            'short_title': short_title
        })
        
        print(f"  ✅ 图表已生成: {file_name}")
        
        # ===================== 7. 保存结果 =====================
        results.append({
            "文件名": file_name,
            "t197": round(t197, 2) if not np.isnan(t197) else "N/A",
            "t195": round(t195, 2) if not np.isnan(t195) else "N/A",
            "trtime": round(trtime, 2) if not np.isnan(trtime) else "N/A",
            "trlt": round(trlt, 2) if not np.isnan(trlt) else "N/A",
            "ltcd": ltcd
        })
        
    except Exception as e:
        print(f"  ❌ 处理失败：{file_name}")
        print(f"     错误详情：{str(e)}")
# ===================== 8. 创建Excel文件并在同一个工作表中汇总所有结果和图表 =====================
if results:
    # 创建结果DataFrame
    result_df = pd.DataFrame(results)
    
    print("\n" + "="*50)
    print("处理结果汇总")
    print("="*50)
    print(result_df)
    
    # 创建Excel工作簿
    workbook = Workbook()
    worksheet = workbook.active
    worksheet.title = "CSV处理结果汇总"
    
    # 设置表头
    headers = list(results[0].keys())
    headers.append("散点图")  # 添加散点图列
    worksheet.append(headers)
    
    # 设置列宽
    column_widths = {
        'A': 40,  # 文件名
        'B': 15,  # t197
        'C': 15,  # t195
        'D': 15,  # trtime
        'E': 15,  # trlt
        'F': 15,  # ltcd
        'G': 30,  # 散点图
    }
    
    for col, width in column_widths.items():
        worksheet.column_dimensions[col].width = width
    
    # 写入数据和插入图表
    for i, (result, chart_data) in enumerate(zip(results, charts_data)):
        # 写入数据行
        row_data = [result[h] for h in list(results[0].keys())]
        worksheet.append(row_data)
        
        # 获取当前行号（从1开始，表头在第1行）
        row_num = i + 2  # 第一行是表头
        
        # 在G列（第7列）插入散点图
        img = Image(io.BytesIO(chart_data['img_data'].getvalue()))
        
        # 设置图片位置和大小
        img.anchor = f'G{row_num}'  # 插入到G列当前行
        
        # 设置图片尺寸（调整为单元格大小）
        img.width = 300  # 宽度，以像素为单位
        img.height = 200  # 高度，以像素为单位
        
        worksheet.add_image(img)
        
        # 设置行高以适应图片
        worksheet.row_dimensions[row_num].height = 150
        
        print(f"  ✅ 已插入图表: {chart_data['file_name']}")
        worksheet.freeze_panes = 'A2'
    # 保存Excel文件
    desktop_path = "/mnt/c/Users/M259941/Desktop/Desktop"
    output_path = os.path.join(desktop_path, "csv处理结果汇总_带图表.xlsx")
    
    workbook.save(output_path)
    print(f"\n✅ Excel文件已保存到: {output_path}")
    print(f"   - 共处理了 {len(results)} 个CSV文件")
    print(f"   - 每个文件的结果和散点图都在同一个工作表中")
    print(f"   - 散点图显示在每行的最后一列")
else:
    print("\n⚠️ 没有找到任何有效数据，请检查：")
    print("1. 文件路径是否正确")
    print("2. CSV文件格式是否正确")
    print("3. 文件是否包含足够的数据行（至少13行）")
    print("4. 文件是否包含足够的列（至少7列）")
